In [ ]:
-- 실행 컨텍스트 설정
USE ROLE ACCOUNTADMIN;
USE DATABASE DEMO;
USE SCHEMA ML_DEMO;

# Module 4: 모델 학습, 실험 추적 및 Model Registry

이 노트북에서는 Snowflake ML을 활용하여 **고객 생애 가치(LTV) 예측 회귀 모델**을 학습하고,
**Model Registry에 등록**하여 버전 관리 및 배포 준비까지 완료합니다.

---

## 비즈니스 목표
고객의 과거 구매 패턴(관찰기간 1992~1997.06)을 기반으로 향후 12개월(1997.07~1998.06)
예상 구매금액(FUTURE_12M_REVENUE)을 예측합니다.

## 이 노트북에서 다루는 내용

**Part 1: 모델 학습 및 실험 추적**
- Snowflake ML `Pipeline`으로 전처리 + 모델 학습
- `ExperimentTracking` API로 실험 run 추적 (RMSE, R² 로깅)
- XGBoost vs Random Forest 성능 비교
- 최적 모델 선택 및 Feature Importance 분석

**Part 2: Model Registry**
- 최적 모델을 Registry에 등록 (`log_model`)
- 버전 관리 (V1, V2)
- Champion/Challenger 교체
- 예측 테스트 및 거버넌스

## 데이터 흐름
```
DEMO.ML_DEMO.CUSTOMER_FEATURES (피처: 관찰기간 / 레이블: 예측 윈도우 12개월 LTV)
        │
        ▼
  Train / Test Split (80:20 고객 랜덤 분할)
        │
        ▼
  Pipeline (OrdinalEncoder → StandardScaler → XGBRegressor / RandomForestRegressor)
        │
        ▼
  Experiment Tracking (RMSE, MAE, R² 비교)
        │
        ▼
  최적 모델 → Model Registry 등록 (CUSTOMER_LTV_PREDICTOR)
        │
        ▼
  Module 5 (Inference) 에서 배포
```


---

# Part 1: 모델 학습 및 실험 추적

XGBoost, Random Forest 등 다양한 알고리즘을 학습하고,
Snowflake ExperimentTracking으로 실험을 추적하여 최적 모델을 선정합니다.

---
## 1. 환경 설정 (Setup)

Container Runtime 환경에서 Snowflake ML 라이브러리와 세션을 초기화합니다.

In [ ]:
from snowflake.snowpark.context import get_active_session
import snowflake.snowpark.functions as F
from snowflake.snowpark.window import Window

# Snowflake ML - 전처리
from snowflake.ml.modeling.preprocessing import StandardScaler, OrdinalEncoder
from snowflake.ml.modeling.pipeline import Pipeline

# Snowflake ML - 모델
from snowflake.ml.modeling.xgboost import XGBRegressor
from snowflake.ml.modeling.ensemble import RandomForestRegressor

# Snowflake ML - 실험 추적
from snowflake.ml.experiment import ExperimentTracking

# 평가 지표
from snowflake.ml.modeling.metrics import (
    roc_auc_score,
)

# 시각화
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
matplotlib.rcParams['axes.unicode_minus'] = False

import warnings
warnings.filterwarnings('ignore')

# 세션 초기화
session = get_active_session()
session.use_database("DEMO")
session.use_schema("ML_DEMO")
session.use_warehouse("COMPUTE_WH")

print(f"Snowpark session: {session.get_current_account()}")
print(f"Database: {session.get_current_database()}")
print(f"Schema:   {session.get_current_schema()}")
print(f"Warehouse:{session.get_current_warehouse()}")
print("\n환경 설정 완료!")from sklearn.metrics import mean_squared_error as sk_mse, r2_score as sk_r2
import numpy as np


---
## 2. 데이터 로드 및 분할

Module 3에서 생성한 `CUSTOMER_FEATURES` 테이블을 로드합니다.  
Snowpark의 **분산 처리** 기반으로 train/test 분할 후 LTV 분포를 확인합니다.

In [ ]:
# 피처 테이블 로드
df = session.table("DEMO.ML_DEMO.CUSTOMER_FEATURES")

print(f"전체 행 수: {df.count():,}")
print(f"컬럼 수:    {len(df.columns)}")
print("\n스키마:")
df.printSchema()

In [ ]:
# 타겟 레이블 및 피처 컬럼 정의
TARGET = "FUTURE_12M_REVENUE"
ID_COL = "C_CUSTKEY"
# 범주형 피처 (OrdinalEncoder 적용)
CATEGORICAL_COLS = ["C_MKTSEGMENT"]
# 수치형 피처 (StandardScaler 적용)
NUMERIC_COLS = [
    "C_ACCTBAL",
    "TOTAL_ORDERS",
    "AVG_ORDER_VALUE",
    "TOTAL_REVENUE",
    "AVG_DISCOUNT",
    "AVG_QUANTITY",
    "DAYS_SINCE_LAST_ORDER",
]
# C_NATIONKEY는 국가코드이지만 NUMBER 타입이므로 수치형으로 처리
# (OrdinalEncoder에 NUMBER 타입을 넣으면 Registry 추론 시 타입 충돌 발생)
NUMERIC_COLS += ["C_NATIONKEY"]
ALL_FEATURES = CATEGORICAL_COLS + NUMERIC_COLS
print(f"타겟 컬럼:   {TARGET}")
print(f"범주형 피처: {CATEGORICAL_COLS}")
print(f"수치형 피처: {NUMERIC_COLS}")
print(f"전체 피처:   {len(ALL_FEATURES)}개")

In [ ]:
# Train / Test 분할 (80/20)
# ⚠️ C_CUSTKEY(ID)는 피처가 아니므로 학습 데이터에서 제외
TRAIN_COLS = ALL_FEATURES + [TARGET]
print(f"학습에 사용할 컬럼: {TRAIN_COLS}")
print(f"제외된 컬럼: C_CUSTKEY (ID), C_NATIONKEY 외 기타")

train_df_raw, test_df_raw = df.randomSplit([0.8, 0.2], seed=42)

# 학습/추론에 사용할 컬럼만 선택 (C_CUSTKEY 제외)
train_df = train_df_raw.select(TRAIN_COLS)
test_df  = test_df_raw.select(TRAIN_COLS)

train_count = train_df.count()
test_count  = test_df.count()
total_count = train_count + test_count

print(f"\n학습 데이터:  {train_count:,} rows ({train_count/total_count*100:.1f}%)")
print(f"테스트 데이터: {test_count:,}  rows ({test_count/total_count*100:.1f}%)")
print(f"컬럼: {train_df.columns}")


In [ ]:
# LTV 분포 확인 (회귀 타겟 — 연속형)
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

ltv_pd = df.select(TARGET).to_pandas()
total = len(ltv_pd)
zero_cnt = (ltv_pd[TARGET] == 0).sum()
pos_cnt  = (ltv_pd[TARGET] > 0).sum()

print("=== 전체 데이터 LTV 분포 ===")
print(f"  $0 (미구매):  {zero_cnt:>7,} ({zero_cnt/total*100:.1f}%)")
print(f"  $0 초과:      {pos_cnt:>7,} ({pos_cnt/total*100:.1f}%)")
print(f"  평균 LTV:     ${ltv_pd[TARGET].mean():>12,.0f}")
print(f"  중앙값 LTV:   ${ltv_pd[TARGET].median():>12,.0f}")
print(f"  최대 LTV:     ${ltv_pd[TARGET].max():>12,.0f}")

# 분포 시각화 (히스토그램)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: 전체 분포
axes[0].hist(ltv_pd[TARGET], bins=60, color='#4C72B0', edgecolor='white')
axes[0].set_title('FUTURE_12M_REVENUE Full Distribution', fontsize=12)
axes[0].set_xlabel('LTV ($)')
axes[0].set_ylabel('Count')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# 오른쪽: $0 초과 분포만
pos_only = ltv_pd[ltv_pd[TARGET] > 0][TARGET]
axes[1].hist(pos_only, bins=60, color='#DD8452', edgecolor='white')
axes[1].set_title('FUTURE_12M_REVENUE > $0 Distribution', fontsize=12)
axes[1].set_xlabel('LTV ($)')
axes[1].set_ylabel('Count')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

print("\n> LTV 분포를 확인하세요. $0 비율이 높으면 zero-inflation을 고려하세요.")


---
## 3. Experiment 생성

Snowflake ML **ExperimentTracking** API는 실험 추적을 Snowflake 내부에서 제공합니다.
`snowflake-ml-python 1.19.0` 이상에서 지원됩니다.

| 개념 | 설명 |
|------|------|
| `ExperimentTracking` | 실험 추적 클라이언트 (세션 기반) |
| `set_experiment(name)` | 실험 그룹 생성/연결 |
| `start_run(run_name)` | 한 번의 학습 실행 시작 |
| `log_metric(key, value)` | 모델 성능 지표 기록 (RMSE, MAE, R² 등) |
| `log_param(key, value)` | 하이퍼파라미터 기록 |

```python
from snowflake.ml.experiment import ExperimentTracking

exp = ExperimentTracking(session=session)
exp.set_experiment("customer_ltv_regression")

run = exp.start_run("xgb_baseline")
# ... 모델 학습 ...
exp.log_metric("rmse", 8500.0)
exp.log_param("n_estimators", 100)
```

> **데이터 저장 위치**: 실험 메타데이터는 `DEMO.ML_DEMO` 스키마 내 Snowflake 오브젝트로 자동 저장됩니다.


In [ ]:
# Experiment 생성 (이미 존재하면 기존 실험을 재사용)
exp = ExperimentTracking(session=session)
exp.set_experiment("customer_ltv_regression")

from datetime import datetime
RUN_SUFFIX = datetime.now().strftime('%H%M%S')
print(f'Run suffix: {RUN_SUFFIX} (재실행 시 고유 이름 생성)')
print(f"Experiment 이름: customer_ltv_regression")
print(f"Database: DEMO / Schema: ML_DEMO")
print("\n실험 추적 준비 완료. 각 run에서 메트릭/파라미터를 로깅합니다.")

---
## 4. 모델 1: XGBoost (기본값)

`Pipeline`으로 **전처리 + 모델 학습**을 하나의 객체로 묶어 관리합니다.

```
Pipeline:
  [1] OrdinalEncoder  ← 범주형 컬럼을 정수로 인코딩
  [2] StandardScaler  ← 수치형 컬럼을 표준화
  [3] XGBRegressor   ← 기본 하이퍼파라미터로 학습
```

In [ ]:
# --- Run 1 시작: XGBoost 기본값 ---
# 재실행 시 이전 run 정리
try:
    exp.end_run()
except Exception:
    pass
run1 = exp.start_run(f"xgboost_baseline_{RUN_SUFFIX}")

# 하이퍼파라미터 정의
XGB_PARAMS_1 = {
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "input_cols": ALL_FEATURES,
    "label_cols": [TARGET],
    "output_cols": ["PREDICTED_LABEL"],
}

# Pipeline 구성
pipeline1 = Pipeline(
    steps=[
        (
            "encoder",
            OrdinalEncoder(
                input_cols=CATEGORICAL_COLS,
                output_cols=CATEGORICAL_COLS,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
        (
            "scaler",
            StandardScaler(
                input_cols=NUMERIC_COLS,
                output_cols=NUMERIC_COLS,
            ),
        ),
        (
            "regressor",
            XGBRegressor(**XGB_PARAMS_1),
        ),
    ]
)

print("Pipeline 구성:")
print("  Step 1: OrdinalEncoder (범주형 → 정수)")
print("  Step 2: StandardScaler (수치형 표준화)")
print("  Step 3: XGBRegressor  (기본 하이퍼파라미터)")
print("\n파이프라인 정의 완료 — 다음 셀에서 .fit() 으로 학습 시작")

In [ ]:
# 학습
pipeline1.fit(train_df)
# 예측
pred1 = pipeline1.predict(test_df)
# 평가
# pandas 변환 후 sklearn 메트릭으로 계산
pred_pd1 = pred1.select(TARGET, "PREDICTED_LABEL").to_pandas()
rmse1 = float(np.sqrt(sk_mse(pred_pd1[TARGET], pred_pd1["PREDICTED_LABEL"])))
r2_1   = float(sk_r2(pred_pd1[TARGET], pred_pd1["PREDICTED_LABEL"]))
print(f"[XGBoost Baseline] 평가 결과")
print(f"  RMSE    : {rmse1:,.0f} (USD)")
print(f"  R²      : {r2_1:.4f}")

In [ ]:
# 메트릭 및 파라미터 로깅
exp.log_metric("rmse", rmse1)
exp.log_metric("r2_score", r2_1)
exp.log_param("model_type", "XGBRegressor")
exp.log_param("n_estimators", XGB_PARAMS_1["n_estimators"])
exp.log_param("max_depth", XGB_PARAMS_1["max_depth"])
exp.log_param("learning_rate", XGB_PARAMS_1["learning_rate"])
exp.log_param("subsample", XGB_PARAMS_1["subsample"])
exp.log_param("colsample_bytree", XGB_PARAMS_1["colsample_bytree"])

# Run 종료


exp.end_run()  # Run 1 종료
print("Run 1 (xgboost_baseline) 로깅 완료 및 종료.")

---
## 5. 모델 2: XGBoost (튜닝)

기본값 대비 다음 하이퍼파라미터를 조정합니다:
- `n_estimators`: 100 → 200 (트리 개수 증가)
- `max_depth`: 6 → 4 (과적합 방지를 위해 깊이 감소)
- `learning_rate`: 0.1 → 0.05 (더 신중한 학습)
- `subsample`: 0.8 → 0.7
- `min_child_weight`: 5 추가 (노이즈 감소)

In [ ]:
# --- Run 2 시작: XGBoost 튜닝 ---
# 재실행 시 이전 run 정리
try:
    exp.end_run()
except Exception:
    pass
run2 = exp.start_run(f"xgboost_tuned_{RUN_SUFFIX}")
XGB_PARAMS_2 = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.7,
    "colsample_bytree": 0.8,
    "min_child_weight": 5,
    "gamma": 0.1,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "random_state": 42,
    "input_cols": ALL_FEATURES,
    "label_cols": [TARGET],
    "output_cols": ["PREDICTED_LABEL"],
}
pipeline2 = Pipeline(
    steps=[
        (
            "encoder",
            OrdinalEncoder(
                input_cols=CATEGORICAL_COLS,
                output_cols=CATEGORICAL_COLS,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
        (
            "scaler",
            StandardScaler(
                input_cols=NUMERIC_COLS,
                output_cols=NUMERIC_COLS,
            ),
        ),
        (
            "regressor",
            XGBRegressor(**XGB_PARAMS_2),
        ),
    ]
)
print("[Run 2] XGBoost 튜닝 - 학습 중...")
pipeline2.fit(train_df)
pred2 = pipeline2.predict(test_df)
# pandas 변환 후 sklearn 메트릭으로 계산
pred_pd2 = pred2.select(TARGET, "PREDICTED_LABEL").to_pandas()
rmse2 = float(np.sqrt(sk_mse(pred_pd2[TARGET], pred_pd2["PREDICTED_LABEL"])))
r2_2   = float(sk_r2(pred_pd2[TARGET], pred_pd2["PREDICTED_LABEL"]))
print(f"\n[XGBoost Tuned] 평가 결과")
print(f"  RMSE    : {rmse2:,.0f} (USD)")
print(f"  R²      : {r2_2:.4f}")
print(f"\n기본값 대비 RMSE 변화: {rmse2 - rmse1:+,.0f}")
print(f"기본값 대비 R² 변화:   {r2_2 - r2_1:+.4f}")

In [ ]:
# 메트릭 및 파라미터 로깅
exp.log_metric("rmse", rmse2)
exp.log_metric("r2_score", r2_2)
exp.log_param("model_type", "XGBRegressor")
exp.log_param("n_estimators", XGB_PARAMS_2["n_estimators"])
exp.log_param("max_depth", XGB_PARAMS_2["max_depth"])
exp.log_param("learning_rate", XGB_PARAMS_2["learning_rate"])
exp.log_param("subsample", XGB_PARAMS_2["subsample"])
exp.log_param("min_child_weight", XGB_PARAMS_2["min_child_weight"])
exp.log_param("gamma", XGB_PARAMS_2["gamma"])
exp.log_param("reg_alpha", XGB_PARAMS_2["reg_alpha"])
exp.log_param("reg_lambda", XGB_PARAMS_2["reg_lambda"])



exp.end_run()  # Run 2 종료
print("Run 2 (xgboost_tuned) 로깅 완료 및 종료.")

---
## 6. 모델 3: Random Forest 비교

XGBoost와 **Random Forest**를 비교하여 앙상블 방식의 차이를 확인합니다.

| 항목 | XGBoost | Random Forest |
|------|---------|---------------|
| 학습 방식 | Boosting (순차적) | Bagging (병렬) |
| 과적합 방지 | 정규화 파라미터 | 트리 개수/깊이 |
| 속도 | 상대적으로 느림 | 빠름 |
| 해석성 | Feature Importance | Feature Importance |

In [ ]:
# --- Run 3 시작: Random Forest ---
# 재실행 시 이전 run 정리
try:
    exp.end_run()
except Exception:
    pass
run3 = exp.start_run(f"random_forest_baseline_{RUN_SUFFIX}")
RF_PARAMS = {
    "n_estimators": 150,
    "max_depth": 10,
    "min_samples_split": 5,
    "min_samples_leaf": 2,
    "random_state": 42,
    "input_cols": ALL_FEATURES,
    "label_cols": [TARGET],
    "output_cols": ["PREDICTED_LABEL"],
}
pipeline3 = Pipeline(
    steps=[
        (
            "encoder",
            OrdinalEncoder(
                input_cols=CATEGORICAL_COLS,
                output_cols=CATEGORICAL_COLS,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
        (
            "scaler",
            StandardScaler(
                input_cols=NUMERIC_COLS,
                output_cols=NUMERIC_COLS,
            ),
        ),
        (
            "regressor",
            RandomForestRegressor(**RF_PARAMS),
        ),
    ]
)
print("[Run 3] Random Forest - 학습 중...")
pipeline3.fit(train_df)
pred3 = pipeline3.predict(test_df)
# pandas 변환 후 sklearn 메트릭으로 계산
pred_pd3 = pred3.select(TARGET, "PREDICTED_LABEL").to_pandas()
rmse3 = float(np.sqrt(sk_mse(pred_pd3[TARGET], pred_pd3["PREDICTED_LABEL"])))
r2_3   = float(sk_r2(pred_pd3[TARGET], pred_pd3["PREDICTED_LABEL"]))
print(f"\n[Random Forest Baseline] 평가 결과")
print(f"  RMSE    : {rmse3:,.0f} (USD)")
print(f"  R²      : {r2_3:.4f}")

In [ ]:
# 메트릭 및 파라미터 로깅
exp.log_metric("rmse", rmse3)
exp.log_metric("r2_score", r2_3)
exp.log_param("model_type", "RandomForestRegressor")
exp.log_param("n_estimators", RF_PARAMS["n_estimators"])
exp.log_param("max_depth", RF_PARAMS["max_depth"])
exp.log_param("min_samples_split", RF_PARAMS["min_samples_split"])
exp.log_param("min_samples_leaf", RF_PARAMS["min_samples_leaf"])



exp.end_run()  # Run 3 종료
print("Run 3 (random_forest_baseline) 로깅 완료 및 종료.")

---
## 7. 실험 결과 비교

세 가지 실험 run의 메트릭을 `exp.list_runs()`로 조회하고,
표 및 차트로 비교하여 **최적 모델**을 선정합니다.

In [ ]:
# 실험 결과 비교 (변수에서 직접 구성)
import pandas as pd

comparison = pd.DataFrame({
    "Run":   ["xgboost_baseline", "xgboost_tuned", "random_forest_baseline"],
    "Model": ["XGBRegressor",    "XGBRegressor",  "RandomForestRegressor"],
    "RMSE":  [round(rmse1, 0),   round(rmse2, 0), round(rmse3, 0)],
    "R²":    [round(r2_1, 4),    round(r2_2, 4),  round(r2_3, 4)],
})

print("=== 전체 실험 Run 비교 ===")
print(comparison.to_string(index=False))

# SQL로 실험 이력 조회 (Snowflake에 저장된 결과)
try:
    runs_sql = session.sql("SHOW RUNS IN EXPERIMENT DEMO.ML_DEMO.customer_ltv_regression").to_pandas()
    print("\n=== Snowflake 저장 Run 이력 ===")
    print(runs_sql.to_string(index=False))
except Exception as e:
    print(f"SQL 조회 오류: {e}")


In [ ]:
# 비교 테이블 수동 구성 (list_runs 컬럼 구조에 따라 조정 가능)
comparison = pd.DataFrame(
    {
        "Run Name": ["xgboost_baseline", "xgboost_tuned", "random_forest_baseline"],
        "Model": ["XGBRegressor", "XGBRegressor", "RandomForestRegressor"],
        "RMSE": [round(rmse1, 4), round(rmse2, 4), round(rmse3, 4)],
        "R²": [round(r2_1, 4), round(r2_2, 4), round(r2_3, 4)],
        "n_estimators": [100, 200, 150],
        "max_depth": [6, 4, 10],
        "learning_rate": [0.1, 0.05, "-"],
    }
)

# 최저 RMSE 기준 하이라이트
best_idx = comparison["R²"].idxmax()
comparison["Best"] = ""
comparison.loc[best_idx, "Best"] = "<-- Best"

print("=== 실험 결과 비교 표 ===")
print(comparison.to_string(index=False))

# 최적 모델 Pipeline 선택
pipelines = [pipeline1, pipeline2, pipeline3]
best_pipeline = pipelines[best_idx]
best_run_name = comparison.loc[best_idx, "Run Name"]
best_is_xgb = "XGBRegressor" in comparison.loc[best_idx, "Model"]

print(f"\n선택된 최적 Pipeline: {best_run_name}")
print(f"XGBoost 계열: {best_is_xgb}")


In [ ]:
# 막대 차트로 비교 시각화
model_names = ["XGB\nBaseline", "XGB\nTuned", "Random\nForest"]
rmse_scores = [rmse1, rmse2, rmse3]
r2_scores   = [r2_1, r2_2, r2_3]

x = np.arange(len(model_names))
width = 0.35

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# RMSE 차트
bars_acc = axes[0].bar(
    x, rmse_scores, width=0.5, color=["#4C72B0", "#DD8452", "#55A868"], edgecolor="white"
)
axes[0].set_ylim(min(rmse_scores) * 0.9, max(rmse_scores) * 1.1)
axes[0].set_title("RMSE Comparison", fontsize=13)
axes[0].set_xticks(x)
axes[0].set_xticklabels(model_names, fontsize=11)
axes[0].set_ylabel("RMSE")
for bar, val in zip(bars_acc, rmse_scores):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.4f}",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

# F1 Score 차트
bars_f1 = axes[1].bar(
    x, r2_scores, width=0.5, color=["#4C72B0", "#DD8452", "#55A868"], edgecolor="white"
)
axes[1].set_ylim(min(r2_scores) - 0.05, 1.0)
axes[1].set_title("F1 Score Comparison", fontsize=13)
axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names, fontsize=11)
axes[1].set_ylabel("R²")
for bar, val in zip(bars_f1, r2_scores):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.002,
        f"{val:.4f}",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

plt.suptitle("Model Performance Comparison\n(LTV Regression)", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\n최적 모델 (RMSE 기준): {comparison.loc[best_idx, 'Run Name']}")
print(f"  RMSE    : {comparison.loc[best_idx, 'RMSE']:,.0f} (USD)")
print(f"  R²      : {comparison.loc[best_idx, 'R²']:.4f}")

In [ ]:
# 최적 모델의 Feature Importance 추출
# XGBoost든 Random Forest든 동일하게 feature_importances_ 사용

best_step = dict(best_pipeline.steps)["regressor"]

if best_is_xgb:
    model = best_step.to_xgboost()
    print(f"Feature Importance 기준 모델: XGBRegressor ({best_run_name})")
else:
    model = best_step.to_sklearn()
    print(f"Feature Importance 기준 모델: RandomForestRegressor ({best_run_name})")

# feature_importances_ (XGBoost, RF 모두 지원)
fi_df = pd.DataFrame({
    "Feature": ALL_FEATURES,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False).reset_index(drop=True)

print("\n=== Feature Importance (Top 10) ===")
print(fi_df.head(10).to_string(index=False))


---
## 8. Feature Importance 시각화

최적 XGBoost 모델에서 **피처 중요도(Feature Importance)**를 추출합니다.

- XGBoost의 `gain` 기반 중요도: 해당 피처로 분기할 때 평균 이득(gain)을 측정
- 중요도가 높은 피처는 분류 결정에 핵심적인 역할을 합니다

In [ ]:
# Top 10 Feature Importance 시각화
top10 = fi_df.head(10).sort_values("Importance", ascending=True)

fig, ax = plt.subplots(figsize=(9, 6))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top10)))
bars = ax.barh(
    top10["Feature"],
    top10["Importance"],
    color=colors,
    edgecolor="white",
    linewidth=0.8,
)

# 값 레이블
for bar, val in zip(bars, top10["Importance"]):
    ax.text(
        bar.get_width() + max(top10["Importance"]) * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.1f}",
        va="center",
        fontsize=10,
    )

ax.set_title(
    f"Top 10 Feature Importance (XGBoost - Gain)\nModel: {best_run_name if best_is_xgb else 'xgboost_tuned'}",
    fontsize=13,
)
ax.set_xlabel("Importance (Gain)")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()

# 주요 인사이트 출력
top_feature = fi_df.iloc[0]["Feature"]
print(f"\n가장 중요한 피처: {top_feature}")
print("\n> Insight: LTV 예측에 가장 영향력 있는 피처를 확인하여")
print("  비즈니스 의사결정 및 추가 피처 엔지니어링에 활용할 수 있습니다.")

In [ ]:
# 최적 모델을 아래 Part 2에서 Registry에 등록합니다.
# Snowflake ML Pipeline은 save_model() 메서드가 없습니다.
# 대신 Model Registry의 log_model()을 사용하여 저장합니다.

print("=== 최적 모델 요약 ===")
print(f"모델: {best_run_name}")
print(f"Pipeline steps: {[name for name, _ in best_pipeline.steps]}")
print(f"RMSE: {min(rmse1, rmse2, rmse3):,.0f} (USD)")
print(f"R²:   {max(r2_1, r2_2, r2_3):.4f}")

print("\n→ 아래 Part 2에서 Model Registry에 등록")
print("  reg.log_model(best_pipeline, model_name='CUSTOMER_LTV_PREDICTOR', ...)")


In [ ]:
# Module 4 최종 요약
print("=" * 60)
print("          Module 4 완료: 모델 학습 및 실험 추적")
print("=" * 60)
print(f"\n실험 실행 수: 3")
print(f"  - Run 1 (xgboost_baseline):       RMSE={rmse1:,.0f} (USD), R²={r2_1:.4f}")
print(f"  - Run 2 (xgboost_tuned):          RMSE={rmse2:,.0f} (USD), R²={r2_2:.4f}")
print(f"  - Run 3 (random_forest_baseline): RMSE={rmse3:,.0f} (USD), R²={r2_3:.4f}")
print(f"\n최적 모델: {best_run_name} (RMSE 기준)")
print("\n아래 Part 2에서 Model Registry에 등록 진행")
print("  → reg.log_model(best_pipeline, model_name='CUSTOMER_LTV_PREDICTOR')")
print("=" * 60)


## Part 1 요약 (모델 학습 및 실험 추적)

이 파트에서 다룬 내용:

| 단계 | 내용 | 주요 API |
|------|------|----------|
| 환경 설정 | 세션·라이브러리 초기화 | `get_active_session()` |
| 데이터 분할 | Train/Test 80:20 분할 | `random_split()` |
| 실험 생성 | 실험 그룹 생성 | `ExperimentTracking`, `set_experiment()` |
| 모델 학습 | XGBoost 기본/튜닝, RF 비교 | `Pipeline`, `XGBRegressor` |
| 실험 추적 | 메트릭·파라미터 기록 | `log_metric()`, `log_param()` |
| 결과 비교 | RMSE/MAE/R² 비교 | `exp.list_runs()` |
| 최적 모델 선정 | RMSE 최소 기준 | `best_pipeline` |

### 다음 단계
**Part 2 (Model Registry)**에서 최적 모델을 등록하고 버전 관리를 수행합니다.


---

# Part 2: Model Registry

Part 1에서 학습된 최적 모델(`best_pipeline`)을 Snowflake Model Registry에 등록하고,
버전 관리·Champion/Challenger 교체·거버넌스를 실습합니다.


In [ ]:
from snowflake.ml.registry import Registry

# Model Registry 초기화
reg = Registry(
    session=session,
    database_name="DEMO",
    schema_name="ML_DEMO"
)

print("Model Registry 초기화 완료")
print(f"Registry 위치: DEMO.ML_DEMO")


# Part 2: Snowflake Model Registry

## 개요

이 모듈에서는 **Snowflake Model Registry**를 활용하여 학습된 모델을 중앙 저장소에 등록하고 관리하는 방법을 학습합니다.

### 학습 목표
- Model Registry 개념 및 필요성 이해
- 학습된 XGBoost 모델 등록 및 버전 관리
- 메타데이터 조회 및 모델 거버넌스 적용
- Champion/Challenger 배포 패턴 구현

### 전제 조건
- Module 4: XGBoost 모델 학습 완료
- `DEMO.ML_DEMO` 스키마에 `CUSTOMER_FEATURES` 테이블 존재

## 1. 환경 설정 (Setup)

필요한 라이브러리를 임포트하고 Snowflake 세션을 초기화합니다.

## 2. Model Registry 초기화

### Snowflake Model Registry란?

**Model Registry**는 ML 모델의 전체 생명주기를 관리하는 중앙화된 저장소입니다.

| 기능 | 설명 |
|------|------|
| **버전 관리** | 모델 버전을 체계적으로 관리 |
| **메타데이터 추적** | 학습 파라미터, 성능 지표 등 기록 |
| **거버넌스** | RBAC 기반 접근 제어 |
| **배포** | 모델을 Snowflake 내에서 직접 서빙 |
| **추적성** | 어떤 데이터로 학습했는지 lineage 추적 |

Snowflake Model Registry는 모델을 **Snowflake 오브젝트**로 저장하므로, 별도의 서버나 스토리지 없이 SQL 권한 체계로 관리할 수 있습니다.

In [ ]:
# 등록된 모델 조회
print("=== 등록된 모델 목록 ===")
for m in reg.models():
    print(f"  {m.name}")

print("\n=== CUSTOMER_LTV_PREDICTOR 버전 목록 ===")
model_ref = reg.get_model("CUSTOMER_LTV_PREDICTOR")
for v in model_ref.versions():
    print(f"  {v.version_name}")


## 3. Part 1에서 학습된 모델 사용

Part 1에서 학습 완료된 `best_pipeline`을 직접 Registry에 등록합니다.

> **전제 조건:** Module 4 노트북을 먼저 실행하여 `best_pipeline`, `train_df`, `test_df` 변수가 메모리에 있어야 합니다.
> 같은 Notebook Service에서 `%run 04_model_training.ipynb`로 실행하거나, 같은 세션에서 순차 실행하세요.


In [ ]:
# Part 1에서 학습된 변수 확인
print("Module 4에서 전달된 변수 확인:")
print(f"  best_pipeline: {type(best_pipeline).__name__}")
print(f"  best_run_name: {best_run_name}")
print(f"  train_df 행 수: {train_df.count():,}")
print(f"  test_df 행 수:  {test_df.count():,}")

# Registry 등록용 변수 설정
pipeline_v1 = best_pipeline
LABEL_COL = "FUTURE_12M_REVENUE"
TARGET = LABEL_COL
print(f"\n등록할 모델: {best_run_name}")


In [ ]:
# V1 모델은 Module 4의 best_pipeline을 직접 사용
# 재학습 불필요 — 이미 학습 완료된 상태
print("V1 모델 준비 완료 (Part 1에서 학습됨)")
print(f"Pipeline steps: {[name for name, _ in pipeline_v1.steps]}")


## 4. 모델 등록 (log_model)

학습된 모델을 Model Registry에 등록합니다. `log_model()`은 모델 오브젝트, 학습에 사용된 데이터 샘플, 메트릭 등을 함께 저장합니다.

### 등록 시 포함되는 정보
- **모델 아티팩트**: 직렬화된 모델 파일
- **입출력 시그니처**: 피처 컬럼명, 데이터 타입
- **메타데이터**: 설명, 태그, 메트릭
- **버전 정보**: 버전 이름 및 생성 시간

In [ ]:
# 기존 모델이 있으면 삭제 (재실행 시 중복 방지)
try:
    reg.delete_model("CUSTOMER_LTV_PREDICTOR")
    print("기존 모델 삭제 완료")
except Exception:
    print("삭제할 기존 모델 없음 - 신규 등록 진행")

In [ ]:
# V1 모델 Registry에 등록
# 재실행 시 기존 버전 삭제 후 재등록
print("V1 모델을 Registry에 등록 중...")

try:
    reg.delete_model("CUSTOMER_LTV_PREDICTOR")
    print("기존 모델 삭제 완료")
except Exception:
    pass  # 없으면 무시

mv_v1 = reg.log_model(
    model=pipeline_v1,
    model_name="CUSTOMER_LTV_PREDICTOR",
    version_name="V1",
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment=f"최적 모델: {best_run_name} (RMSE={min(rmse1,rmse2,rmse3):,.0f} (USD), R²={max(r2_1,r2_2,r2_3):.4f})"
)

print("V1 모델 등록 완료!")
print(f"모델명: CUSTOMER_LTV_PREDICTOR")
print(f"버전: V1")


In [ ]:
# 별칭 설정: champion으로 지정
try:
    mv_v1.set_alias("champion")
    print("V1 별칭 설정: 'champion'")
except Exception as e:
    print(f"별칭 설정 스킵 (이미 존재): {e}")

# 설명 추가
mv_v1.description = "고객 LTV 예측 회귀 모델 V1 - Part 1 최적 모델 (best_pipeline)"
print("모델 설명 업데이트 완료")

# 메트릭 로깅 (학습 RMSE)
from sklearn.metrics import mean_squared_error as _mse
import numpy as np
train_preds = pipeline_v1.predict(train_df)
train_pred_pd = train_preds.select(LABEL_COL, "PREDICTED_LABEL").to_pandas()
train_rmse = float(np.sqrt(_mse(train_pred_pd[LABEL_COL], train_pred_pd["PREDICTED_LABEL"])))

mv_v1.set_metric("train_rmse", train_rmse)
print(f"\n학습 RMSE 메트릭 저장: {train_rmse:,.0f} (USD)")


## 5. 모델 버전 관리

동일한 모델에 여러 버전을 등록하여 성능 비교 및 점진적 업데이트를 관리할 수 있습니다.

### 버전 관리의 필요성
- **A/B 테스트**: 기존 버전(Champion) vs 신규 버전(Challenger) 비교
- **롤백**: 문제 발생 시 이전 버전으로 즉시 복구
- **감사**: 각 시점의 모델 상태 추적
- **재현성**: 동일한 모델로 언제든지 예측 재현 가능

In [ ]:
# V2 모델 학습 (다른 하이퍼파라미터로 Champion 도전)
print("XGBoost 모델 학습 중 (V2 - 향상된 파라미터)...")

pipeline_v2 = Pipeline(
    steps=[
        (
            "encoder",
            OrdinalEncoder(
                input_cols=CATEGORICAL_COLS,
                output_cols=CATEGORICAL_COLS,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
        ),
        (
            "scaler",
            StandardScaler(
                input_cols=NUMERIC_COLS,
                output_cols=NUMERIC_COLS,
            ),
        ),
        (
            "regressor",
            XGBRegressor(
                input_cols=ALL_FEATURES,
                label_cols=[LABEL_COL],
                output_cols=["PREDICTED_LABEL"],
                n_estimators=200,
                max_depth=6,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42
            )
        )
    ]
)

pipeline_v2.fit(train_df)
print("V2 모델 학습 완료")


In [ ]:
# V2 모델 Registry에 등록
print("V2 모델을 Registry에 등록 중...")

mv_v2 = reg.log_model(
    model=pipeline_v2,
    model_name="CUSTOMER_LTV_PREDICTOR",
    version_name="V2",
    target_platforms=["WAREHOUSE", "SNOWPARK_CONTAINER_SERVICES"],
    comment="XGBoost V2 (n_estimators=200, max_depth=6, lr=0.05, subsample=0.8)"
)

# 별칭 설정: challenger
try:
    mv_v2.set_alias("challenger")
    print("V2 별칭 설정: 'challenger'")
except Exception as e:
    print(f"별칭 설정 스킵: {e}")

mv_v2.description = "고객 LTV 예측 회귀 모델 V2 - 하이퍼파라미터 튜닝 버전"

# V2 RMSE 계산 (sklearn)
from sklearn.metrics import mean_squared_error as _mse
import numpy as np
train_preds_v2 = pipeline_v2.predict(train_df)
v2_pred_pd = train_preds_v2.select(LABEL_COL, "PREDICTED_LABEL").to_pandas()
train_rmse_v2 = float(np.sqrt(_mse(v2_pred_pd[LABEL_COL], v2_pred_pd["PREDICTED_LABEL"])))

mv_v2.set_metric("train_rmse", train_rmse_v2)

print(f"\nV2 모델 등록 완료!")
print(f"V2 학습 RMSE: {train_rmse_v2:,.0f} (USD)")


In [ ]:
# V1 vs V2 메타데이터 비교
print("=== V1 vs V2 메타데이터 비교 ===")

comparison_data = {
    "항목": [
        "버전",
        "별칭 (Alias)",
        "n_estimators",
        "max_depth",
        "learning_rate",
        "subsample",
        "train_rmse"
    ],
    "V1 (Champion)": [
        "V1",
        "champion",
        100,
        4,
        0.1,
        1.0,
        f"{train_rmse:.4f}"
    ],
    "V2 (Challenger)": [
        "V2",
        "challenger",
        200,
        6,
        0.05,
        0.8,
        f"{train_rmse_v2:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.to_string(index=False))

## 6. 모델 메타데이터 조회

Registry에 등록된 모델의 상세 메타데이터를 조회합니다. 이는 모델 감사(audit) 및 재현성 보장에 중요합니다.

In [ ]:
# V1 모델 상세 정보 조회
print("=== V1 모델 메타데이터 ===")

mv_v1_ref = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V1")

# 모델 설명 출력
print(f"설명: {mv_v1_ref.description}")

# 메트릭 조회
print("\n--- 저장된 메트릭 ---")
try:
    metrics = mv_v1_ref.get_metric("train_rmse")
    print(f"train_rmse: {metrics}")
except Exception as e:
    print(f"메트릭 조회: {e}")

In [ ]:
# 입출력 시그니처 확인
print("=== V1 모델 함수 목록 ===")
functions = mv_v1_ref.show_functions()
for func in functions:
    if isinstance(func, dict):
        print(f"  함수명: {func.get('name', func)}")
    else:
        print(f"  함수: {func}")

print("\n=== 모델 입출력 시그니처 ===")
print(f"입력 피처: {ALL_FEATURES}")
print(f"레이블 컬럼: {LABEL_COL}")
print(f"출력 컬럼: PREDICTED_LABEL")


In [ ]:
# 전체 모델 정보 출력 (SQL을 통한 조회)
print("=== SQL을 통한 모델 조회 ===")
model_info_sql = """
SHOW VERSIONS IN MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR
"""
try:
    model_info = session.sql(model_info_sql).to_pandas()
    print(model_info.to_string(index=False))
except Exception as e:
    print(f"SQL 조회 오류: {e}")

## 7. 등록된 모델로 예측 테스트

Registry에서 모델을 불러와 예측을 수행합니다. 이 방식은 모델 서빙의 표준 패턴입니다.

### 핵심 포인트
- Registry에서 불러온 모델은 **Snowflake 내에서 실행**됩니다
- 데이터를 외부로 이동할 필요 없이 Snowflake 컴퓨팅 활용
- `function_name="predict"` 또는 `function_name="predict"` 선택 가능

In [ ]:
# Registry에서 V1 모델 불러오기
print("Registry에서 V1 모델 로드 중...")
mv_loaded = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V1")
print(f"모델 로드 완료: CUSTOMER_LTV_PREDICTOR / V1")

# 테스트 데이터 준비 (학습 시 사용한 컬럼과 동일해야 함)
test_sample = test_df.limit(100)
print(f"테스트 샘플 수: {test_sample.count()}")
print(f"테스트 컬럼: {test_sample.columns}")


In [ ]:
# 등록된 모델로 예측 실행
print("=== V1 모델로 예측 실행 ===")
predictions = mv_loaded.run(
    test_sample,
    function_name="predict"
)

# 예측 결과 샘플 출력
print("\n예측 결과 샘플 (상위 10개):")
result_df = predictions.select(
    ALL_FEATURES[:3] + [LABEL_COL, "PREDICTED_LABEL"]
).limit(10).to_pandas()

print(result_df.to_string(index=False))

In [ ]:
# 테스트 데이터 예측 결과 요약 (RMSE)
from sklearn.metrics import mean_squared_error as _mse, r2_score as _r2
import numpy as np

pred_pd = predictions.select(LABEL_COL, "PREDICTED_LABEL").to_pandas()
test_rmse = float(np.sqrt(_mse(pred_pd[LABEL_COL], pred_pd["PREDICTED_LABEL"])))
test_r2 = float(_r2(pred_pd[LABEL_COL], pred_pd["PREDICTED_LABEL"]))

print("=== 테스트 데이터 예측 결과 요약 ===")
print(f"테스트 샘플 수: {len(pred_pd)}")
print(f"RMSE: {test_rmse:,.0f} (USD)")
print(f"R²:   {test_r2:.4f}")

# 예측값 기술통계
print("\n예측값 통계:")
print(pred_pd["PREDICTED_LABEL"].describe().to_string())


## 8. 모델 거버넌스

Snowflake Model Registry는 Snowflake의 **RBAC(Role-Based Access Control)** 체계를 그대로 활용합니다. 이를 통해 모델 접근을 세밀하게 제어할 수 있습니다.

### 거버넌스 체계

```
ACCOUNTADMIN
    └── SYSADMIN
        ├── ML_ENGINEER_ROLE  (모델 생성/수정/삭제)
        ├── DATA_SCIENTIST_ROLE  (모델 읽기/예측)
        └── ANALYST_ROLE  (예측 결과만 조회)
```

### 권한 종류

| 권한 | 설명 |
|------|------|
| `USAGE` | 모델 조회 및 예측 실행 |
| `MONITOR` | 모델 메타데이터 조회 |
| `MODIFY` | 모델 업데이트 (설명, 태그) |
| `OWNERSHIP` | 모델 삭제 포함 전체 권한 |

In [ ]:
# 모델 거버넌스 - 권한 부여 예시
# 실제 환경에서는 적절한 역할에 권한을 부여합니다

governance_examples = """
-- 1. 데이터 사이언티스트 역할에 모델 사용 권한 부여
GRANT USAGE ON MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR
    TO ROLE DATA_SCIENTIST_ROLE;

-- 2. 특정 버전에 대한 접근 제어
GRANT USAGE ON MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR
    TO ROLE ANALYST_ROLE;

-- 3. ML 엔지니어에게 수정 권한 부여
GRANT MODIFY ON MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR
    TO ROLE ML_ENGINEER_ROLE;

-- 4. 현재 모델 권한 확인
SHOW GRANTS ON MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR;
"""

print("=== 모델 거버넌스 SQL 예시 ===")
print(governance_examples)

In [ ]:
# 현재 모델 권한 상태 조회 (실제 실행)
print("=== 현재 CUSTOMER_LTV_PREDICTOR 권한 현황 ===")
try:
    grants = session.sql(
        "SHOW GRANTS ON MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR"
    ).to_pandas()
    if len(grants) > 0:
        print(grants[["privilege", "granted_on", "name", "grantee_name"]].to_string(index=False))
    else:
        print("현재 명시적 권한 없음 (소유자 전용)")
except Exception as e:
    print(f"권한 조회 오류: {e}")

print("\n거버넌스 모범 사례:")
print("  1. 최소 권한 원칙 (Principle of Least Privilege) 적용")
print("  2. 프로덕션 모델은 별도 역할로 접근 제어")
print("  3. 모델 수정 권한은 ML 엔지니어에게만 부여")
print("  4. 감사 로그를 통한 모델 접근 이력 추적")

## 9. Champion/Challenger 패턴

### Champion/Challenger 패턴이란?

프로덕션 환경에서 새로운 모델 버전을 안전하게 배포하기 위한 전략입니다.

```
┌─────────────────────────────────────────────┐
│              프로덕션 트래픽                  │
│                    │                         │
│         ┌──────────┴──────────┐              │
│         ▼                     ▼              │
│  ┌─────────────┐    ┌─────────────────┐     │
│  │  Champion   │    │   Challenger    │     │
│  │     V1      │    │      V2         │     │
│  │  (90% 트래픽)│    │  (10% 트래픽)   │     │
│  └─────────────┘    └─────────────────┘     │
│         │                     │              │
│         └──────────┬──────────┘              │
│                    ▼                         │
│              성능 비교 → 프로모션             │
└─────────────────────────────────────────────┘
```

### 운영 절차
1. 새 버전을 **Challenger**로 등록
2. 소량의 트래픽으로 성능 검증
3. 성능이 우수하면 **Champion으로 승격**
4. 기존 Champion은 **retired** 또는 삭제

In [ ]:
# 현재 Champion/Challenger 상태 확인
print("=== 현재 Champion/Challenger 상태 ===")

model_ref = reg.get_model("CUSTOMER_LTV_PREDICTOR")

for version_name in ["V1", "V2"]:
    try:
        v = model_ref.version(version_name)
        print(f"\n버전: {version_name}")
        print(f"  설명: {v.description}")
    except Exception as e:
        print(f"버전 {version_name} 조회 오류: {e}")

# SQL로 alias 확인
print("\n=== 버전 별칭 (SQL 조회) ===")
session.sql("SHOW VERSIONS IN MODEL DEMO.ML_DEMO.CUSTOMER_LTV_PREDICTOR").show()


In [ ]:
# Champion vs Challenger 성능 비교
from sklearn.metrics import mean_squared_error as _mse, r2_score as _r2
import numpy as np

print("=== Champion (V1) vs Challenger (V2) 성능 비교 ===")

eval_data = test_df.limit(200)

# V1 (Champion) 예측
mv_champ = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V1")
champ_preds = mv_champ.run(eval_data, function_name="predict")
champ_pd = champ_preds.select(LABEL_COL, "PREDICTED_LABEL").to_pandas()
champ_rmse = float(np.sqrt(_mse(champ_pd[LABEL_COL], champ_pd["PREDICTED_LABEL"])))
champ_r2 = float(_r2(champ_pd[LABEL_COL], champ_pd["PREDICTED_LABEL"]))

# V2 (Challenger) 예측
mv_chall = reg.get_model("CUSTOMER_LTV_PREDICTOR").version("V2")
chall_preds = mv_chall.run(eval_data, function_name="predict")
chall_pd = chall_preds.select(LABEL_COL, "PREDICTED_LABEL").to_pandas()
chall_rmse = float(np.sqrt(_mse(chall_pd[LABEL_COL], chall_pd["PREDICTED_LABEL"])))
chall_r2 = float(_r2(chall_pd[LABEL_COL], chall_pd["PREDICTED_LABEL"]))

print(f"Champion  (V1): RMSE = {champ_rmse:,.0f} (USD), R² = {champ_r2:.4f}")
print(f"Challenger(V2): RMSE = {chall_rmse:,.0f} (USD), R² = {chall_r2:.4f}")

# RMSE 낮을수록 좋음
if chall_rmse < champ_rmse:
    print(f"\n✅ Challenger가 RMSE {champ_rmse - chall_rmse:,.0f} (USD) 더 우수합니다.")
    print("→ V2를 Champion으로 승격 권장")
else:
    print(f"\n현재 Champion이 더 우수하거나 동등합니다.")
    print("→ 기존 Champion(V1) 유지")


In [ ]:
# Champion 승격 시나리오 (Challenger → Champion)
print("=== Champion 승격 시나리오 ===")
print("V2 성능이 우수하다고 가정하고 Champion으로 승격합니다.\n")

promote_example = """
# 1단계: V2를 Champion으로 승격
mv_v2.set_alias("champion")

# 2단계: V1의 Champion 별칭 제거 (자동으로 교체됨)
#         Snowflake에서는 동일 별칭을 재설정하면 이전 설정이 교체됩니다

# 3단계: V1을 retired로 표시
mv_v1.description = "[RETIRED] 이전 Champion - V2로 대체됨"

print("Champion 승격 완료: V1 → V2")
"""

print("승격 코드 예시:")
print(promote_example)

print("\n현재 최종 상태:")
print("  V1: champion (현재 운영 중)")
print("  V2: challenger (검증 중)")
print("\n모델 거버넌스 완료!")

## Part 2 요약 (Model Registry)

이 모듈에서 다룬 내용:

| 단계 | 내용 | 주요 API |
|------|------|----------|
| Registry 초기화 | 중앙 모델 저장소 연결 | `Registry(session, database_name, schema_name)` |
| 모델 등록 | 학습된 모델 저장 | `reg.log_model(model, model_name, version_name)` |
| 버전 관리 | 다중 버전 관리 | `reg.list_models()`, `model.list_versions()` |
| 메타데이터 | 설명·메트릭 기록 | `mv.description`, `mv.set_metric()` |
| 예측 | Registry 모델로 추론 | `mv.run(data, function_name="predict")` |
| 거버넌스 | RBAC 접근 제어 | `GRANT USAGE ON MODEL` |
| Champion/Challenger | 안전한 배포 패턴 | `mv.set_alias("champion")` |

### 다음 단계
**Module 5 (Inference)**에서는 등록된 모델을 활용하여 배치/실시간 예측을 수행하는 방법을 학습합니다.